# Research Lab: измерять, а не угадывать

Activation и gradient norms, causal leakage и сравнение изменений.

> Все примеры этого урока работают на CPU, не скачивают модели и используют искусственные данные. Они объясняют механизм; качество настоящего обучения проверяется отдельно.

## Что важно понять

Падающий train loss не доказывает корректность модели. Проверяйте конечность градиентов, зависимость от входа, отсутствие доступа к будущему и качество на validation. Сравнивайте изменения при одинаковых данных, seed и числе шагов.

Ниже тест причинности: меняем будущие токены и проверяем, изменился ли ранний output. Для causal модели изменение должно быть близко к нулю. Research Lab выполняет дополнительные probes и анализ архитектуры; маленький CPU probe не заменяет полноценный benchmark.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Запустите эксперимент

Запускайте ячейки по порядку. Код можно менять; Shift + Enter выполняет текущую ячейку.

In [ ]:
block = nn.TransformerEncoderLayer(16, 4, 32, dropout=0, batch_first=True).eval()
original = torch.randn(2, 6, 16)
modified = original.clone(); modified[:, 3:] += torch.randn_like(modified[:, 3:]) * 5
causal = torch.triu(torch.ones(6, 6, dtype=torch.bool), diagonal=1)
with torch.no_grad():
    masked_delta = (block(original, src_mask=causal)[:, :3] - block(modified, src_mask=causal)[:, :3]).abs().max()
    unmasked_delta = (block(original)[:, :3] - block(modified)[:, :3]).abs().max()
print('causal future-token delta:', masked_delta.item())
print('unmasked future-token delta:', unmasked_delta.item())
assert masked_delta < 1e-5
block.train(); loss = block(original, src_mask=causal).mul(original).mean(); loss.backward()
for name, parameter in block.named_parameters():
    if parameter.grad is not None:
        print(name, 'gradient norm:', parameter.grad.norm().item())

## Попробуйте сами

Уберите causal mask и убедитесь, что тест на masked_delta больше не проходит. Добавьте dropout: почему для сравнения выходов нужен eval или контролируемый RNG?

Сохраните копию ноутбука и запишите, что изменили, чего ожидали и что измерили.

## Продолжить в студии

Кнопка «Открыть раздел студии» над уроком ведёт к связанным инструментам. Учебный код сам не запускает большие тренировки. Перед реальным запуском проверьте данные, ресурсы и поддержку метода у выбранной модели.

## Источники

- [PyTorch profiler](https://docs.pytorch.org/tutorials/recipes/recipes/profiler_recipe.html)